# iNaturalist Cleaned Real-Trajectory Reproduction

This CPU-only notebook reconstructs the real-data analysis used in the manuscript. It starts from public iNaturalist identification histories, reconstructs the manuscript's monotone compatible trajectories, applies the trajectory-quality audit, and evaluates the hierarchy-only, final-horizon, and full-history identified sets.

## Analysis reproduced here

The notebook:

1. queries public iNaturalist observations from class **Aves** in the manuscript observation-created window (2022-01-01 through 2024-12-31);
2. retrieves identification histories and taxonomic metadata;
3. selects a supported induced hierarchy of up to eight genera and five species per genus;
4. retains observations whose **first compatible, non-withdrawn identification is genuinely coarse** (genus/family/order/class) and defines that event as time zero;
5. reconstructs a monotone compatible envelope at horizons 0/1/7/30/90 days;
6. audits incompatible and withdrawn identifications and applies the manuscript species-level quality rule (compatible-event rate at least 85% and withdrawn-event rate at most 20%);
7. evaluates the hierarchy-only, final-horizon, and all-horizon sensitivity regions over \(\Gamma\in\{6,8,12,20\}\);
8. computes the empirical raw-feasibility thresholds \(\Gamma_{\min,\mathrm{all}}\) and \(\Gamma_{\min,\mathrm{final}}\);
9. evaluates the principal unrelaxed setting \(\Gamma=12\) and repeats it on the unfiltered 695-observation cohort as a robustness analysis.

No images are downloaded and no predictive model is trained. The target is population identification, not classification accuracy.

## Important modeling choice

iNaturalist identification records can disagree, be withdrawn, or move sideways in the taxonomy. Reconstructing the platform's historical consensus taxon would require reproducing platform-specific consensus logic. The manuscript therefore uses a **monotone compatible envelope**:

- the observation's current species assignment defines the root-to-species reference lineage;
- an identification is compatible when its taxon lies on that lineage;
- withdrawn and outside-lineage identifications are retained for auditing but are not used as trajectory states;
- time zero is the first compatible, non-withdrawn coarse identification;
- at each later horizon, the state is the deepest compatible model node reached by that time.

This is deliberately not presented as a reconstruction of the historical iNaturalist community taxon.

> **Reproducibility note.** iNaturalist metadata and taxonomy are live resources. Historical identifications, current taxon assignments, and taxonomy mappings can change after the manuscript analysis. The repository therefore stores the validated manuscript values under `results/paper/real_data/` as an immutable audit record, while this notebook provides the executable API-based reconstruction pipeline.



### API basis

The current iNaturalist API exposes observation search, observation details, and identification records. Observation-details responses include an `identifications` array, and taxon objects expose ancestor IDs. The observation-details endpoint accepts comma-delimited observation IDs (up to 200), which lets this notebook retrieve histories efficiently in batches.


In [ ]:
# Core imports and configuration
import json, math, time, zipfile, os, re
from pathlib import Path
from collections import defaultdict, Counter
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.optimize import linprog

API = "https://api.inaturalist.org/v1"
HEADERS = {
    "User-Agent": "ISHF-Reproducibility/1.0 (public metadata research)"
}

# -------- MANUSCRIPT ANALYSIS CONFIGURATION --------
TARGET_TAXON_QUERY = "Aves"
TARGET_TAXON_RANK = "class"

# Observation-created window used in the manuscript.
CREATED_D1 = "2022-01-01"
CREATED_D2 = "2024-12-31"

# Metadata-only candidate pool before automatic genus/species selection.
TARGET_N_OBSERVATIONS = 4000

# Time is measured from the first compatible, non-withdrawn coarse identification.
HORIZON_DAYS = [0, 1, 7, 30, 90]
COARSE_RANKS = {"genus", "family", "order", "class"}

# Automatic induced hierarchy used for the manuscript cohort construction.
MAX_GENERA = 8
MAX_SPECIES_PER_GENUS = 5
MIN_OBS_PER_SPECIES = 15
MIN_SPECIES_PER_GENUS = 2

# Sensitivity path reported in the manuscript.
GAMMA_GRID = [6.0, 8.0, 12.0, 20.0]
PRINCIPAL_GAMMA = 12.0

# API politeness / robustness.
REQUEST_SLEEP = 0.25
MAX_RETRIES = 5
REQUEST_TIMEOUT = 60

OUTDIR = Path("inaturalist_cleaned_reproduction_results")
OUTDIR.mkdir(exist_ok=True)

print("Target:", TARGET_TAXON_QUERY, f"({TARGET_TAXON_RANK})")
print("Created window:", CREATED_D1, "to", CREATED_D2)
print("Candidate observations requested:", TARGET_N_OBSERVATIONS)
print("Horizons from first compatible coarse identification:", HORIZON_DAYS)
print("Hierarchy selection: max genera =", MAX_GENERA, ", max species/genus =", MAX_SPECIES_PER_GENUS)


## 1. API helpers and preflight

In [ ]:

session = requests.Session()
session.headers.update(HEADERS)

def api_get(path, params=None, timeout=REQUEST_TIMEOUT):
    url = f"{API}/{path.lstrip('/')}"
    last_err = None

    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, params=params, timeout=timeout)

            if resp.status_code == 429:
                wait = min(30, 2 ** attempt)
                print(f"Rate-limited; sleeping {wait}s")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()

        except Exception as e:
            last_err = e
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(min(15, 2 ** attempt))

    raise RuntimeError(last_err)

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

# Resolve the target taxon.
taxon_search = api_get(
    "taxa/autocomplete",
    params={
        "q": TARGET_TAXON_QUERY,
        "rank": TARGET_TAXON_RANK,
        "per_page": 10,
    },
)

candidates = taxon_search.get("results", [])
if not candidates:
    raise RuntimeError(
        f"Could not resolve target taxon {TARGET_TAXON_QUERY!r}"
    )

target_taxon = candidates[0]
TARGET_TAXON_ID = int(target_taxon["id"])

print("Resolved target taxon:")
print({
    "id": TARGET_TAXON_ID,
    "name": target_taxon.get("name"),
    "rank": target_taxon.get("rank"),
    "preferred_common_name": target_taxon.get("preferred_common_name"),
})


## 2. Retrieve current species-level research-grade observations


In [ ]:

def fetch_observation_index(target_n):
    results = []
    id_above = None

    while len(results) < target_n:
        params = {
            "taxon_id": TARGET_TAXON_ID,
            "quality_grade": "research",
            "rank": "species",
            "created_d1": CREATED_D1,
            "created_d2": CREATED_D2,
            "per_page": 200,
            "order_by": "id",
            "order": "asc",
            "verifiable": "true",
        }

        if id_above is not None:
            params["id_above"] = id_above

        data = api_get("observations", params=params)
        batch = data.get("results", [])

        if not batch:
            break

        results.extend(batch)
        id_above = max(int(o["id"]) for o in batch)

        print(f"Indexed {len(results):,} observations...")
        time.sleep(REQUEST_SLEEP)

        if len(batch) < 200:
            break

    return results[:target_n]

obs_index = fetch_observation_index(TARGET_N_OBSERVATIONS)

if len(obs_index) < 200:
    raise RuntimeError(
        f"Only {len(obs_index)} observations were returned. "
        "Broaden the date window or target taxon."
    )

obs_ids = [int(o["id"]) for o in obs_index]

print("Observation IDs indexed:", len(obs_ids))
print("ID range:", min(obs_ids), "to", max(obs_ids))



The API's observation-details endpoint is used next because its response explicitly includes the complete `identifications` array for each returned observation.


In [ ]:

def fetch_observation_details(ids):
    full = []

    for bi, batch in enumerate(chunks(ids, 200), start=1):
        id_path = ",".join(str(x) for x in batch)
        data = api_get(f"observations/{id_path}")
        got = data.get("results", [])
        full.extend(got)

        print(
            f"Detail batch {bi}: {len(got)} returned; "
            f"cumulative {len(full)}"
        )
        time.sleep(REQUEST_SLEEP)

    return full

observations = fetch_observation_details(obs_ids)

if len(observations) < 0.95 * len(obs_ids):
    raise RuntimeError(
        "Too many indexed observations were missing from the detail fetch."
    )

print("Full observations retrieved:", len(observations))

has_identifications = sum(
    isinstance(o.get("identifications"), list)
    for o in observations
)
print("Observations with an identifications array:", has_identifications)


## 3. Build taxonomic metadata for final species and ancestors

In [ ]:

# Collect all final species and their ancestor IDs.
final_species = {}
all_taxon_ids = set([TARGET_TAXON_ID])

for o in observations:
    tax = o.get("taxon") or {}
    if tax.get("id") is None:
        continue

    sid = int(tax["id"])
    final_species[sid] = tax
    all_taxon_ids.add(sid)

    for aid in tax.get("ancestor_ids", []) or []:
        all_taxon_ids.add(int(aid))

print("Unique final species in raw sample:", len(final_species))
print("Unique ancestor/species taxon IDs:", len(all_taxon_ids))


In [ ]:

def fetch_taxa_details(ids):
    out = {}

    # /taxa/{id1,id2,...} supports multi-ID lookup; use conservative chunks.
    for batch in chunks(sorted(set(ids)), 30):
        path = "taxa/" + ",".join(str(x) for x in batch)
        data = api_get(path)

        for t in data.get("results", []):
            out[int(t["id"])] = t

        time.sleep(REQUEST_SLEEP)

    return out

taxa = fetch_taxa_details(all_taxon_ids)

print("Taxon records fetched:", len(taxa))

def lineage_ids_from_observation(o):
    tax = o.get("taxon") or {}
    return [int(x) for x in (tax.get("ancestor_ids") or [])] + [int(tax["id"])]

def rank_ancestor_id(o, desired_rank):
    lineage = lineage_ids_from_observation(o)

    matches = [
        tid for tid in lineage
        if taxa.get(tid, {}).get("rank") == desired_rank
    ]

    return matches[-1] if matches else None

# Attach final genus.
rows = []

for o in observations:
    tax = o.get("taxon") or {}
    if tax.get("id") is None:
        continue

    sid = int(tax["id"])
    gid = rank_ancestor_id(o, "genus")

    rows.append({
        "observation_id": int(o["id"]),
        "species_id": sid,
        "species_name": taxa.get(sid, tax).get("name"),
        "genus_id": gid,
        "genus_name": taxa.get(gid, {}).get("name") if gid else None,
    })

taxon_frame = pd.DataFrame(rows)

print(taxon_frame.head().to_string(index=False))
print("Observations with a resolved genus:",
      int(taxon_frame.genus_id.notna().sum()))


## 4. Automatically choose a well-supported induced hierarchy

In [ ]:

species_counts = (
    taxon_frame.dropna(subset=["genus_id"])
    .groupby(["genus_id","genus_name","species_id","species_name"])
    .size()
    .reset_index(name="n")
)

eligible_species = species_counts[
    species_counts.n >= MIN_OBS_PER_SPECIES
].copy()

genus_support = (
    eligible_species.groupby(["genus_id","genus_name"])
    .agg(
        eligible_species=("species_id","nunique"),
        eligible_observations=("n","sum"),
    )
    .reset_index()
)

eligible_genera = genus_support[
    genus_support.eligible_species >= MIN_SPECIES_PER_GENUS
].sort_values(
    ["eligible_observations","eligible_species"],
    ascending=False
).head(MAX_GENERA)

if len(eligible_genera) < 2:
    raise RuntimeError(
        "Fewer than two genera meet the support thresholds. "
        "Increase TARGET_N_OBSERVATIONS, broaden the date range, "
        "or reduce MIN_OBS_PER_SPECIES."
    )

selected_species_rows = []

for _, grow in eligible_genera.iterrows():
    gid = grow.genus_id

    sub = eligible_species[
        eligible_species.genus_id == gid
    ].sort_values("n", ascending=False).head(MAX_SPECIES_PER_GENUS)

    selected_species_rows.append(sub)

selected_species_df = pd.concat(
    selected_species_rows,
    ignore_index=True
)

SELECTED_SPECIES = set(
    selected_species_df.species_id.astype(int)
)
SELECTED_GENERA = set(
    selected_species_df.genus_id.astype(int)
)

selected_species_df.to_csv(
    OUTDIR/"selected_species.csv",
    index=False
)

print("Selected genera:", len(SELECTED_GENERA))
print("Selected species:", len(SELECTED_SPECIES))
print(selected_species_df.to_string(index=False))


## 5. Reconstruct monotone compatible trajectories

In [ ]:
obs_by_id = {int(o["id"]): o for o in observations}

selected_obs_ids = set(
    taxon_frame[
        taxon_frame.species_id.isin(SELECTED_SPECIES)
    ].observation_id.astype(int)
)

selected_observations = [
    obs_by_id[i]
    for i in selected_obs_ids
    if i in obs_by_id
]

print("Selected observations before coarse-start restriction:", len(selected_observations))

def parse_time(x):
    return pd.to_datetime(x, utc=True, errors="coerce")

def final_lineage(o):
    tax = o["taxon"]
    return [int(x) for x in (tax.get("ancestor_ids") or [])] + [int(tax["id"])]

def canonical_model_state(taxon_id, final_sid, final_gid, lineage):
    # Three-level induced model: ROOT -> selected genus -> selected species.
    tid = int(taxon_id)

    if tid == final_sid:
        return f"S:{final_sid}", 2

    # Anything on the lineage at / below the final genus but above species
    # is conservatively collapsed to the genus node.
    if final_gid in lineage:
        genus_depth = lineage.index(final_gid)
        if tid in lineage and lineage.index(tid) >= genus_depth:
            return f"G:{final_gid}", 1

    # Coarser compatible labels collapse to ROOT.
    return "ROOT", 0

trajectory_rows = []
event_rows = []

for o in selected_observations:
    oid = int(o["id"])
    observation_created_at = parse_time(o.get("created_at"))

    sid = int(o["taxon"]["id"])
    gid = rank_ancestor_id(o, "genus")
    lineage = final_lineage(o)
    lineage_set = set(lineage)

    ids = list(o.get("identifications") or [])
    ids = sorted(ids, key=lambda z: parse_time(z.get("created_at")))

    # `state_events` contains only compatible, non-withdrawn identifications.
    # Withdrawn and outside-lineage records remain in `obs_event_rows` for auditing.
    state_events = []
    obs_event_rows = []
    compatible_count = 0
    incompatible_count = 0
    withdrawn_count = 0
    raw_depths = []

    for ident in ids:
        tax = ident.get("taxon") or {}
        if tax.get("id") is None:
            continue

        itid = int(tax["id"])
        itime = parse_time(ident.get("created_at"))
        if pd.isna(itime):
            continue

        current = ident.get("current")
        rank = str(tax.get("rank") or "").lower()
        compatible = itid in lineage_set

        if current is False:
            withdrawn_count += 1

        if compatible:
            compatible_count += 1
            state, depth = canonical_model_state(itid, sid, gid, lineage)
            if current is not False:
                state_events.append((itime, state, depth, itid, current, rank))
                raw_depths.append(depth)
        else:
            incompatible_count += 1

        obs_event_rows.append({
            "observation_id": oid,
            "identification_id": ident.get("id"),
            "identification_created_at": itime,
            "identification_taxon_id": itid,
            "identification_taxon_name": tax.get("name"),
            "identification_rank": rank,
            "current": current,
            "compatible_with_current_species_lineage": compatible,
        })

    if not state_events:
        continue

    state_events.sort(key=lambda x: x[0])
    first_time, first_state, first_depth, _, _, first_rank = state_events[0]

    # Manuscript coarse-start restriction: the first usable compatible
    # identification itself must be genuinely coarse, not species-resolved.
    if first_rank not in COARSE_RANKS or first_depth >= 2:
        continue

    time_zero = first_time

    # Raw usable compatible events can still become coarser after becoming finer;
    # we audit that fact, while the analysis trajectory uses the monotone envelope.
    raw_nonmonotone = any(
        b < a for a, b in zip(raw_depths[:-1], raw_depths[1:])
    )

    states = []
    for d in HORIZON_DAYS:
        cutoff = time_zero + pd.Timedelta(days=d)
        deepest_state, deepest_depth = "ROOT", 0
        for itime, state, depth, itid, current, rank in state_events:
            if time_zero <= itime <= cutoff and depth > deepest_depth:
                deepest_state, deepest_depth = state, depth
        states.append(deepest_state)

    trajectory_rows.append({
        "observation_id": oid,
        "observation_created_at": observation_created_at,
        "time_zero_first_compatible_coarse_id": time_zero,
        "species_id": sid,
        "species_name": taxa.get(sid, o["taxon"]).get("name"),
        "genus_id": gid,
        "genus_name": taxa.get(gid, {}).get("name"),
        "n_identifications": len(ids),
        "n_compatible_identifications": compatible_count,
        "n_incompatible_identifications": incompatible_count,
        "n_withdrawn_identifications": withdrawn_count,
        "raw_compatible_sequence_nonmonotone": raw_nonmonotone,
        **{
            f"state_{d}d": state
            for d, state in zip(HORIZON_DAYS, states)
        }
    })
    event_rows.extend(obs_event_rows)

traj_df = pd.DataFrame(trajectory_rows)
events_df = pd.DataFrame(event_rows)

traj_df.to_csv(OUTDIR/"reconstructed_trajectories_initial.csv", index=False)
events_df.to_csv(OUTDIR/"identification_event_audit_initial.csv", index=False)

print("Coarse-start trajectories reconstructed:", len(traj_df))
print("Species:", traj_df.species_id.nunique() if len(traj_df) else 0)
print("Genera:", traj_df.genus_id.nunique() if len(traj_df) else 0)
if len(traj_df):
    print(traj_df.head().to_string(index=False))


## Taxonomy-quality cleanup used in the manuscript

During trajectory auditing, *Larus smithsonianus* showed a markedly anomalous identification history in the validated manuscript run. The analysis therefore applies a species-level quality rule excluding a species when its compatible-identification-event rate is below 85% or its withdrawn-identification-event rate exceeds 20%. In the archived manuscript run, only *Larus smithsonianus* met this rule, removing 56 observations from the initial 695-observation coarse-start cohort and leaving 639 observations, 21 species, and eight genera.

The rule is documented here as part of the final analysis pipeline and arose from trajectory auditing in the validated manuscript run.


In [ ]:
# Species-level quality audit and cleanup.
COMPATIBLE_RATE_MIN = 0.85
WITHDRAWN_RATE_MAX = 0.20
EXPECTED_EXCLUDED_SPECIES = {"Larus smithsonianus"}

obs_species = (
    traj_df[["observation_id", "species_id", "species_name"]]
    .drop_duplicates()
)

event_aug = events_df.merge(
    obs_species,
    on="observation_id",
    how="inner",
)

species_quality = (
    event_aug
    .groupby(["species_id", "species_name"], dropna=False)
    .agg(
        total_identifications=("identification_id", "count"),
        compatible_identifications=("compatible_with_current_species_lineage", "sum"),
        withdrawn_identifications=("current", lambda x: int((x == False).sum())),
    )
    .reset_index()
)

species_quality["compatible_event_rate"] = (
    species_quality.compatible_identifications /
    species_quality.total_identifications
)
species_quality["withdrawn_event_rate"] = (
    species_quality.withdrawn_identifications /
    species_quality.total_identifications
)
species_quality["exclude"] = (
    (species_quality.compatible_event_rate < COMPATIBLE_RATE_MIN) |
    (species_quality.withdrawn_event_rate > WITHDRAWN_RATE_MAX)
)

species_quality.to_csv(OUTDIR / "species_quality_audit.csv", index=False)

excluded_species_ids = set(
    species_quality.loc[species_quality.exclude, "species_id"].astype(int)
)
excluded_species_names = set(
    species_quality.loc[species_quality.exclude, "species_name"].astype(str)
)

# Keep the full coarse-start cohort for the manuscript robustness analysis.
traj_df_initial = traj_df.copy().reset_index(drop=True)
events_df_initial = event_aug.copy().reset_index(drop=True)

initial_n = len(traj_df_initial)
initial_species = traj_df_initial.species_id.nunique()
initial_genera = traj_df_initial.genus_id.nunique()

traj_df = traj_df_initial[
    ~traj_df_initial.species_id.astype(int).isin(excluded_species_ids)
].copy().reset_index(drop=True)

events_df = events_df_initial[
    ~events_df_initial.species_id.astype(int).isin(excluded_species_ids)
].copy().reset_index(drop=True)

traj_df.to_csv(OUTDIR / "reconstructed_trajectories.csv", index=False)
events_df.to_csv(OUTDIR / "identification_event_audit.csv", index=False)

print("Initial coarse-start cohort:", initial_n, "observations,", initial_species, "species,", initial_genera, "genera")
print("Excluded species:", sorted(excluded_species_names))
print("Cleaned cohort:", len(traj_df), "observations,", traj_df.species_id.nunique(), "species,", traj_df.genus_id.nunique(), "genera")

if excluded_species_names != EXPECTED_EXCLUDED_SPECIES:
    print(
        "WARNING: the current live iNaturalist data/taxonomy no longer reproduce "
        "the exact manuscript exclusion set. Compare against results/paper/real_data/."
    )


## 6. Trajectory-quality audit

In [ ]:

total_ids = int(events_df.shape[0])
compatible_ids = int(events_df.compatible_with_current_species_lineage.sum())
withdrawn_ids = int((events_df.current == False).sum())

compatible_rate = (
    compatible_ids / total_ids if total_ids else np.nan
)

withdrawn_rate = (
    withdrawn_ids / total_ids if total_ids else np.nan
)

obs_with_incompatible = float(
    np.mean(traj_df.n_incompatible_identifications > 0)
)

obs_raw_nonmonotone = float(
    traj_df.raw_compatible_sequence_nonmonotone.mean()
)

quality_summary = pd.DataFrame([{
    "selected_observations": len(traj_df),
    "selected_genera": traj_df.genus_id.nunique(),
    "selected_species": traj_df.species_id.nunique(),
    "total_identifications": total_ids,
    "compatible_identification_rate": compatible_rate,
    "withdrawn_identification_rate": withdrawn_rate,
    "observation_fraction_with_any_incompatible_id": obs_with_incompatible,
    "observation_fraction_raw_compatible_nonmonotone": obs_raw_nonmonotone,
}])

quality_summary.to_csv(
    OUTDIR/"trajectory_quality_summary.csv",
    index=False
)

print(quality_summary.to_string(index=False))

for d in HORIZON_DAYS:
    print(f"\nState counts at {d} days")
    print(traj_df[f"state_{d}d"].value_counts().head(20).to_string())



## 7. Build the induced hierarchy

The model hierarchy is

\[
\text{ROOT}
\rightarrow
\{\text{selected genera}\}
\rightarrow
\{\text{selected species within each genus}\}.
\]

Labels above genus are collapsed to ROOT. Compatible labels below genus but above species are collapsed to their genus. This avoids pretending that sparsely represented intermediate taxonomic ranks are separate well-supported children.


In [ ]:

species_to_genus = (
    traj_df[["species_id","genus_id"]]
    .drop_duplicates()
    .set_index("species_id")
    .genus_id.astype(int)
    .to_dict()
)

species_ids = sorted(traj_df.species_id.astype(int).unique())
genus_ids = sorted(traj_df.genus_id.astype(int).unique())

leaf_index = {
    sid:i for i,sid in enumerate(species_ids)
}
nL = len(species_ids)

children = {
    "ROOT": [f"G:{gid}" for gid in genus_ids]
}

for gid in genus_ids:
    children[f"G:{gid}"] = [
        f"S:{sid}"
        for sid in species_ids
        if species_to_genus[sid] == gid
    ]

def subtree_species(node):
    if node.startswith("S:"):
        return [int(node.split(":")[1])]

    if node.startswith("G:"):
        gid = int(node.split(":")[1])
        return [
            sid for sid in species_ids
            if species_to_genus[sid] == gid
        ]

    if node == "ROOT":
        return list(species_ids)

    raise KeyError(node)

def subtree_vector(node):
    v = np.zeros(nL)
    for sid in subtree_species(node):
        v[leaf_index[sid]] = 1.0
    return v

print("Internal nodes:", len(children))
print("Leaves:", nL)
for u, ch in children.items():
    print(u, "->", ch)


## 8. Observable crossing masses and retrospective true proportions

In [ ]:
N = len(traj_df)

def state_in_subtree(state, child):
    state = str(state)

    if child.startswith("G:"):
        gid = int(child.split(":")[1])

        if state == child:
            return True

        if state.startswith("S:"):
            sid = int(state.split(":")[1])
            return species_to_genus.get(sid) == gid

        return False

    if child.startswith("S:"):
        return state == child

    return state == "ROOT"

r = defaultdict(lambda: defaultdict(dict))

for u, chs in children.items():
    for c in chs:
        for d in HORIZON_DAYS:
            r[u][c][d] = float(
                np.mean([
                    state_in_subtree(s, c)
                    for s in traj_df[f"state_{d}d"]
                ])
            )

# Current-species empirical proportions are used only for a retrospective
# diagnostic of branch-entry heterogeneity; they are not supplied to the
# identified-set optimization.
p_reference = np.zeros(nL)

for sid, count in traj_df.species_id.value_counts().items():
    p_reference[leaf_index[int(sid)]] = count / N

assert abs(p_reference.sum() - 1.0) < 1e-12

crossing_rows = []

for u, chs in children.items():
    for c in chs:
        for d in HORIZON_DAYS:
            crossing_rows.append({
                "parent": u,
                "child": c,
                "horizon_days": d,
                "crossing_mass": r[u][c][d],
                "current_species_reference_subtree_mass": float(
                    subtree_vector(c) @ p_reference
                )
            })

crossing_df = pd.DataFrame(crossing_rows)
crossing_df.to_csv(OUTDIR/"crossing_masses.csv", index=False)

print(crossing_df.head(20).to_string(index=False))


## 9. Diagnostic empirical refinement-odds heterogeneity

In [ ]:
or_rows = []

for u, chs in children.items():
    for d in HORIZON_DAYS:
        q = {}

        for c in chs:
            s = float(subtree_vector(c) @ p_reference)
            q[c] = r[u][c][d] / s if s > 0 else np.nan

        for j in chs:
            for k in chs:
                if j >= k:
                    continue

                qj, qk = q[j], q[k]

                if (
                    np.isfinite(qj)
                    and np.isfinite(qk)
                    and 0 < qj < 1
                    and 0 < qk < 1
                ):
                    oj = qj/(1-qj)
                    ok = qk/(1-qk)
                    pair_or = max(oj/ok, ok/oj)
                else:
                    pair_or = np.inf

                or_rows.append({
                    "parent": u,
                    "child_j": j,
                    "child_k": k,
                    "horizon_days": d,
                    "q_j": qj,
                    "q_k": qk,
                    "symmetric_odds_ratio_reference_diagnostic": pair_or,
                })

or_df = pd.DataFrame(or_rows)
or_df.to_csv(
    OUTDIR/"retrospective_refinement_odds_audit.csv",
    index=False
)

finite_or = or_df[
    np.isfinite(or_df.symmetric_odds_ratio_reference_diagnostic)
]

print("Finite sibling OR diagnostic count:", len(finite_or))
if len(finite_or):
    print(
        finite_or.symmetric_odds_ratio_reference_diagnostic
        .describe(percentiles=[.5,.75,.9,.95,.99])
        .to_string()
    )


## 10. Compact sensitivity LP

In [ ]:
FINAL_H = HORIZON_DAYS[-1]

def build_constraints(gamma=None, horizons=None, eta=0.0):
    if horizons is None:
        horizons = HORIZON_DAYS

    A = []
    b = []

    # Hierarchy compatibility using the final observed horizon.
    for u, chs in children.items():
        for c in chs:
            M_c = r[u][c][FINAL_H]
            A.append(-subtree_vector(c))
            b.append(-M_c)

    if gamma is not None:
        for u, chs in children.items():
            vv = {c: subtree_vector(c) for c in chs}

            for d in horizons:
                for j in chs:
                    for k in chs:
                        if j == k:
                            continue

                        rj = r[u][j][d]
                        rk = r[u][k][d]

                        A.append(rj*vv[k] - gamma*rk*vv[j])
                        b.append((1.0-gamma)*rj*rk + eta)

    return np.asarray(A), np.asarray(b)

Aeq = np.ones((1, nL))
beq = np.array([1.0])
bounds = [(0, None)]*nL

def lp_feasible(A, b):
    res = linprog(
        np.zeros(nL),
        A_ub=A,
        b_ub=b,
        A_eq=Aeq,
        b_eq=beq,
        bounds=bounds,
        method="highs"
    )
    return bool(res.success)

def solve_bounds(A, b):
    lows, highs = [], []

    for k in range(nL):
        c = np.zeros(nL)
        c[k] = 1.0

        lo = linprog(c, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")
        hi = linprog(-c, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=beq, bounds=bounds, method="highs")

        if not (lo.success and hi.success):
            return None

        lows.append(lo.fun)
        highs.append(-hi.fun)

    return np.asarray(lows), np.asarray(highs)

def minimum_relaxation(gamma, horizons):
    A0, b0 = build_constraints(gamma=None, horizons=horizons, eta=0.0)
    Ag, bg = build_constraints(gamma=gamma, horizons=horizons, eta=0.0)

    # Separate compatibility rows from sensitivity rows.
    ncompat = len(A0)
    As = Ag[ncompat:]
    bs = bg[ncompat:]

    # Variables are p plus the common relaxation eta.
    A_aug = []
    b_aug = []

    for row, rhs in zip(A0, b0):
        A_aug.append(np.r_[row, 0.0])
        b_aug.append(rhs)

    for row, rhs in zip(As, bs):
        A_aug.append(np.r_[row, -1.0])
        b_aug.append(rhs)

    c = np.zeros(nL+1)
    c[-1] = 1.0

    res = linprog(
        c,
        A_ub=np.asarray(A_aug),
        b_ub=np.asarray(b_aug),
        A_eq=np.r_[np.ones(nL), 0.0][None, :],
        b_eq=[1.0],
        bounds=[(0, None)]*(nL+1),
        method="highs"
    )

    if not res.success:
        return np.nan

    return float(res.x[-1])

def raw_feasible(gamma, horizons):
    A, b = build_constraints(gamma=gamma, horizons=horizons, eta=0.0)
    return lp_feasible(A, b)

def minimum_raw_feasible_gamma(horizons, lower=1.0, upper=32.0, tol=1e-6, max_iter=80):
    """Smallest Gamma (to numerical tolerance) for which the unrelaxed system is feasible."""
    if raw_feasible(lower, horizons):
        return float(lower)

    hi = float(upper)
    while not raw_feasible(hi, horizons):
        hi *= 2.0
        if hi > 1e6:
            return np.nan

    lo = float(lower)
    for _ in range(max_iter):
        mid = 0.5*(lo + hi)
        if raw_feasible(mid, horizons):
            hi = mid
        else:
            lo = mid
        if hi - lo <= tol:
            break
    return float(hi)

# Hierarchy-only baseline.
A_hierarchy, b_hierarchy = build_constraints(gamma=None)
hierarchy_bounds = solve_bounds(A_hierarchy, b_hierarchy)

if hierarchy_bounds is None:
    raise RuntimeError("Hierarchy-compatibility polytope is infeasible.")

hierarchy_mean_width = float(np.mean(hierarchy_bounds[1]-hierarchy_bounds[0]))
print("Hierarchy-only mean leaf interval width:", hierarchy_mean_width)


## 11. Sensitivity path: feasibility, interval width, and temporal gain


In [ ]:
profile_rows = []
bound_tables = []

for gamma in GAMMA_GRID:
    eta_all = minimum_relaxation(gamma, HORIZON_DAYS)
    eta_final = minimum_relaxation(gamma, [FINAL_H])

    # Use the common full-history relaxation for the temporal comparison.
    A_all, b_all = build_constraints(
        gamma=gamma,
        horizons=HORIZON_DAYS,
        eta=eta_all + 1e-12
    )
    A_final, b_final = build_constraints(
        gamma=gamma,
        horizons=[FINAL_H],
        eta=eta_all + 1e-12
    )

    # Final-horizon model under its own minimum relaxation: diagnostic only.
    A_final_own, b_final_own = build_constraints(
        gamma=gamma,
        horizons=[FINAL_H],
        eta=eta_final + 1e-12
    )

    all_bounds = solve_bounds(A_all, b_all)
    final_bounds = solve_bounds(A_final, b_final)
    final_own_bounds = solve_bounds(A_final_own, b_final_own)

    if all_bounds is None or final_bounds is None:
        continue

    all_width = all_bounds[1]-all_bounds[0]
    final_width = final_bounds[1]-final_bounds[0]

    profile_rows.append({
        "gamma": gamma,
        "raw_feasible_full_history": bool(raw_feasible(gamma, HORIZON_DAYS)),
        "raw_feasible_final_horizon": bool(raw_feasible(gamma, [FINAL_H])),
        "eta_all_horizons": eta_all,
        "eta_final_horizon": eta_final,
        "hierarchy_only_mean_width": hierarchy_mean_width,
        "final_mean_width_common_relaxation": float(final_width.mean()),
        "final_mean_width_own_relaxation": (
            float((final_own_bounds[1]-final_own_bounds[0]).mean())
            if final_own_bounds is not None else np.nan
        ),
        "all_mean_width": float(all_width.mean()),
        "all_tightening_vs_hierarchy": float(
            1.0 - all_width.mean()/hierarchy_mean_width
        ),
        "all_tightening_vs_final": float(
            1.0 - all_width.mean()/final_width.mean()
            if final_width.mean() > 0 else 0.0
        ),
        "strictly_tightened_leaves_vs_final": int(
            np.sum(all_width < final_width-1e-8)
        ),
    })

    tmp = pd.DataFrame({
        "species_id": species_ids,
        "species_name": [taxa.get(sid, {}).get("name", str(sid)) for sid in species_ids],
        "lower": all_bounds[0],
        "upper": all_bounds[1],
        "width": all_width,
        "gamma": gamma,
    })
    bound_tables.append(tmp)

profile_df = pd.DataFrame(profile_rows)
bounds_df = pd.concat(bound_tables, ignore_index=True)

profile_df.to_csv(OUTDIR/"gamma_profile.csv", index=False)
bounds_df.to_csv(OUTDIR/"leaf_bounds_by_gamma.csv", index=False)

print(profile_df.to_string(index=False))


## 12. Raw-feasibility thresholds and principal setting


In [ ]:
gamma_min_all = minimum_raw_feasible_gamma(HORIZON_DAYS)
gamma_min_final = minimum_raw_feasible_gamma([FINAL_H])

print("Current-run raw-feasibility threshold, all horizons:", gamma_min_all)
print("Current-run raw-feasibility threshold, final horizon:", gamma_min_final)

principal_row = profile_df[np.isclose(profile_df.gamma, PRINCIPAL_GAMMA)]
if principal_row.empty:
    raise RuntimeError(f"Principal Gamma={PRINCIPAL_GAMMA:g} was not evaluated.")
principal_row = principal_row.iloc[0]

if not bool(principal_row.raw_feasible_full_history):
    print(
        "WARNING: the current live API rerun does not make the principal Gamma=12 "
        "full-history system raw-feasible. Compare with the archived manuscript record."
    )

threshold_record = {
    "gamma_min_all_current_run": float(gamma_min_all),
    "gamma_min_final_current_run": float(gamma_min_final),
    "principal_gamma": PRINCIPAL_GAMMA,
}
(OUTDIR/"raw_feasibility_thresholds.json").write_text(
    json.dumps(threshold_record, indent=2)
)


## 13. Cleaned-cohort cross-check and unfiltered-cohort robustness


In [ ]:
def analyze_trajectory_cohort(traj_source, gamma=12.0):
    """Standalone unrelaxed analysis used for the 695-observation robustness check."""
    df = traj_source.copy().reset_index(drop=True)
    local_species_to_genus = (
        df[["species_id", "genus_id"]]
        .drop_duplicates()
        .set_index("species_id")
        .genus_id.astype(int)
        .to_dict()
    )
    local_species = sorted(df.species_id.astype(int).unique())
    local_genera = sorted(df.genus_id.astype(int).unique())
    li = {sid: i for i, sid in enumerate(local_species)}
    L = len(local_species)

    local_children = {"ROOT": [f"G:{gid}" for gid in local_genera]}
    for gid in local_genera:
        local_children[f"G:{gid}"] = [
            f"S:{sid}" for sid in local_species
            if local_species_to_genus[sid] == gid
        ]

    def sub_species(node):
        if node.startswith("S:"):
            return [int(node.split(":")[1])]
        if node.startswith("G:"):
            gid = int(node.split(":")[1])
            return [sid for sid in local_species if local_species_to_genus[sid] == gid]
        return list(local_species)

    def vec(node):
        out = np.zeros(L)
        for sid in sub_species(node):
            out[li[sid]] = 1.0
        return out

    def state_in(state, child):
        state = str(state)
        if child.startswith("G:"):
            gid = int(child.split(":")[1])
            if state == child:
                return True
            if state.startswith("S:"):
                return local_species_to_genus.get(int(state.split(":")[1])) == gid
            return False
        if child.startswith("S:"):
            return state == child
        return state == "ROOT"

    rr = defaultdict(lambda: defaultdict(dict))
    for u, chs in local_children.items():
        for c in chs:
            for h in HORIZON_DAYS:
                rr[u][c][h] = float(np.mean([state_in(s, c) for s in df[f"state_{h}d"]]))

    Aeq_local = np.ones((1, L))
    bounds_local = [(0, None)]*L

    def constraints(g=None, hs=None):
        if hs is None:
            hs = HORIZON_DAYS
        A, b = [], []
        for u, chs in local_children.items():
            for c in chs:
                A.append(-vec(c))
                b.append(-rr[u][c][FINAL_H])
        if g is not None:
            for u, chs in local_children.items():
                vv = {c: vec(c) for c in chs}
                for h in hs:
                    for j in chs:
                        for k in chs:
                            if j == k:
                                continue
                            rj, rk = rr[u][j][h], rr[u][k][h]
                            A.append(rj*vv[k] - g*rk*vv[j])
                            b.append((1.0-g)*rj*rk)
        return np.asarray(A), np.asarray(b)

    def feasible(g, hs):
        A, b = constraints(g, hs)
        res = linprog(np.zeros(L), A_ub=A, b_ub=b, A_eq=Aeq_local, b_eq=[1.0], bounds=bounds_local, method="highs")
        return bool(res.success)

    def threshold(hs, lo=1.0, hi=32.0, tol=1e-6):
        if feasible(lo, hs):
            return float(lo)
        while not feasible(hi, hs):
            hi *= 2.0
            if hi > 1e6:
                return np.nan
        for _ in range(80):
            mid = 0.5*(lo+hi)
            if feasible(mid, hs):
                hi = mid
            else:
                lo = mid
            if hi-lo <= tol:
                break
        return float(hi)

    def widths(g=None, hs=None):
        A, b = constraints(g, hs)
        low, high = [], []
        for k in range(L):
            c = np.zeros(L); c[k] = 1.0
            a = linprog(c, A_ub=A, b_ub=b, A_eq=Aeq_local, b_eq=[1.0], bounds=bounds_local, method="highs")
            z = linprog(-c, A_ub=A, b_ub=b, A_eq=Aeq_local, b_eq=[1.0], bounds=bounds_local, method="highs")
            if not (a.success and z.success):
                return None
            low.append(a.fun); high.append(-z.fun)
        return np.asarray(high)-np.asarray(low)

    wh = widths(None, HORIZON_DAYS)
    wf = widths(gamma, [FINAL_H])
    wa = widths(gamma, HORIZON_DAYS)

    out = {
        "observations": int(len(df)),
        "species": int(df.species_id.nunique()),
        "genera": int(df.genus_id.nunique()),
        "gamma": float(gamma),
        "gamma_min_all": threshold(HORIZON_DAYS),
        "gamma_min_final": threshold([FINAL_H]),
        "raw_feasible_at_gamma": feasible(gamma, HORIZON_DAYS),
    }
    if wh is not None and wf is not None and wa is not None:
        out.update({
            "hierarchy_only_mean_width": float(wh.mean()),
            "final_mean_width": float(wf.mean()),
            "all_mean_width": float(wa.mean()),
            "tightening_vs_hierarchy": float(1.0-wa.mean()/wh.mean()),
            "tightening_vs_final": float(1.0-wa.mean()/wf.mean()),
            "strictly_tightened_species": int(np.sum(wa < wf-1e-8)),
        })
    return out

cleaned_crosscheck = analyze_trajectory_cohort(traj_df, PRINCIPAL_GAMMA)
unfiltered_robustness = analyze_trajectory_cohort(traj_df_initial, PRINCIPAL_GAMMA)

print("Cleaned cohort current-run cross-check:")
print(json.dumps(cleaned_crosscheck, indent=2))
print("\nUnfiltered cohort current-run robustness:")
print(json.dumps(unfiltered_robustness, indent=2))

summary = {
    "stage": "inaturalist_cleaned_reproduction",
    "analysis_target": "population identification under informative hierarchical refinement",
    "target_taxon_query": TARGET_TAXON_QUERY,
    "target_taxon_id": TARGET_TAXON_ID,
    "created_d1": CREATED_D1,
    "created_d2": CREATED_D2,
    "candidate_observations_requested": TARGET_N_OBSERVATIONS,
    "horizon_days": HORIZON_DAYS,
    "coarse_ranks": sorted(COARSE_RANKS),
    "trajectory_definition": "monotone compatible envelope from first compatible non-withdrawn coarse identification",
    "lineage_reference": "current species assignment returned by iNaturalist at retrieval time",
    "quality_rule": {
        "compatible_event_rate_min": COMPATIBLE_RATE_MIN,
        "withdrawn_event_rate_max": WITHDRAWN_RATE_MAX,
        "excluded_species_current_run": sorted(excluded_species_names),
    },
    "initial_coarse_start_current_run": {
        "observations": int(initial_n),
        "species": int(initial_species),
        "genera": int(initial_genera),
    },
    "cleaned_cohort_current_run": {
        "observations": int(len(traj_df)),
        "species": int(traj_df.species_id.nunique()),
        "genera": int(traj_df.genus_id.nunique()),
        "compatible_event_rate": float(compatible_rate),
        "withdrawn_event_rate": float(withdrawn_rate),
    },
    "gamma_grid": GAMMA_GRID,
    "gamma_min_all_current_run": float(gamma_min_all),
    "gamma_min_final_current_run": float(gamma_min_final),
    "principal_gamma": PRINCIPAL_GAMMA,
    "cleaned_principal_current_run": cleaned_crosscheck,
    "unfiltered_principal_current_run": unfiltered_robustness,
    "reproducibility_note": (
        "iNaturalist is live. Current-run values may differ from the archived manuscript values. "
        "Use results/paper/real_data/reported_real_data_summary.json to verify the manuscript."
    ),
}

with open(OUTDIR/"inaturalist_reproduction_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved reproduction summary.")


## 14. Diagnostic plots and result archive


In [ ]:
plt.figure(figsize=(7.4, 4.8))
plt.axhline(hierarchy_mean_width, linestyle="--", label="Hierarchy only")
plt.plot(profile_df.gamma, profile_df.final_mean_width_common_relaxation, marker="s", label="Final horizon")
plt.plot(profile_df.gamma, profile_df.all_mean_width, marker="o", label="All horizons")
if np.isfinite(gamma_min_all):
    plt.axvline(gamma_min_all, linestyle=":", label=r"$\Gamma_{\min,\mathrm{all}}$")
plt.xlabel("Gamma")
plt.ylabel("Mean species-level identified width")
plt.title("iNaturalist sensitivity path")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR/"sensitivity_path.png", dpi=160)
plt.close()

plt.figure(figsize=(7.4, 4.8))
plt.plot(profile_df.gamma, profile_df.eta_all_horizons, marker="o", label="All horizons")
plt.plot(profile_df.gamma, profile_df.eta_final_horizon, marker="s", label="Final horizon only")
plt.xlabel("Gamma")
plt.ylabel("Minimum relaxation eta")
plt.title("Empirical sensitivity-polytope feasibility")
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR/"minimum_relaxation_path.png", dpi=160)
plt.close()

zip_name = "inaturalist_cleaned_reproduction_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTDIR.iterdir():
        zf.write(p, arcname=p.name)

print("Result archive:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    pass
